<a href="https://colab.research.google.com/github/But-Vladyslava/Machine-Learning-Uni-/blob/main/%D0%9B%D0%A0_9_%D0%91%D1%83%D1%82_3_12_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Бут Владислава Євгенівна

**Група:** 3-12

**Дата:**  12.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
n_users = ratings_df['user_id'].nunique()
n_items = ratings_df['item_id'].nunique()

min_rating = ratings_df['rating'].min()
max_rating = ratings_df['rating'].max()
mean_rating = ratings_df['rating'].mean()

missing_values = ratings_df.isnull().sum()

print(f"Кількість унікальних користувачів: {n_users}")
print(f"Кількість унікальних фільмів: {n_items}")
print(f"Мінімальний рейтинг: {min_rating}")
print(f"Максимальний рейтинг: {max_rating}")
print(f"Середній рейтинг: {mean_rating:.2f}")
print("\nПропущені значення по колонках:")
print(missing_values)

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальний рейтинг: 1.0
Максимальний рейтинг: 5.0
Середній рейтинг: 3.53

Пропущені значення по колонках:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

Набір даних містить інформацію про від 943 унікальних користувачів про 1682 фільми. Оцінки користувачів знаходяться в діапазоні від 1.0 до 5.0, при цьому середній рейтинг становить приблизно 3.53, що вказує на загалом позитивну тенденцію в оцінюванні фільмів. Пропущені значення у наборі даних відсутні, що забезпечує його повноту для подальшого аналізу та побудови рекомендаційних систем.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(
    ratings_df, test_size=0.2, random_state=42
)

In [6]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']], reader
)
# з train_data у навчальний набір Surprise (trainset)
trainset = train_data.build_full_trainset()

# форм. testset як список кортежів для Surprise
testset = list(
    zip(test_df['user_id'], test_df['item_id'], test_df['rating'])
)

print(f'Кількість оцінок у тренувальному наборі: {trainset.n_ratings}')
print(f'Кількість оцінок у тестовому наборі: {len(testset)}')

Кількість оцінок у тренувальному наборі: 80000
Кількість оцінок у тестовому наборі: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [7]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

In [8]:
# власн. варіант

param_grid_svd = {
    'n_factors': [50, 100],
    'n_epochs': [20, 30],  # додано к-сть епох навчання
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05, 0.1],  # додано сильнішу регуляризацію
}

param_grid_knn = {
    'k': [20, 40, 60],  # додано більше значення k для згладжування
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson', 'msd'],  # додано метрику MSD
        'user_based': [False, True],
    },
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

Для моделі SVD обрала значення навколо базових налаштувань бібліотеки Surprise (n_factors: 50 і 100 задають оптимальну складність латентного простору; n_epochs: 20 і 30 забезпечують достатню кількість ітерацій для збіжності; lr_all: 0.005 і 0.010 дають стабільний крок градієнтного спуску без розбіжності, а reg_all від 0.02 до 0.1 контролює перенавчання на розріджених даних). Для KNNBasic кількість сусідів k [20, 40, 60] - для балансу між локальною точністю та згладжуванням викидів, min_k [1, 5] відсікає ненадійні прогнози при малій кількості оцінок, а sim_options перевіряє обидва підходи (user_based та item_based) на трьох ключових метриках схожості (cosine, pearson, msd). Також подібні значення часто використвують розробники і модель не перенавчається, тобто ці діапазони можна назвати "робочими".

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [9]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
gs_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
gs_svd.fit(train_data)

# найкращі результати для SVD
print("--- SVD Best Results ---")
print("Найкращий RMSE:", gs_svd.best_score['rmse'])
print("Найкращі параметри для RMSE:", gs_svd.best_params['rmse'])

--- SVD Best Results ---
Найкращий RMSE: 0.9303076413508009
Найкращі параметри для RMSE: {'n_factors': 100, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}


In [11]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
gs_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
gs_knn.fit(train_data)

# найкращі результати для KNNBasic
print("\n--- KNNBasic Best Results ---")
print("Найкращий RMSE:", gs_knn.best_score['rmse'])
print("Найкращі параметри для RMSE:", gs_knn.best_params['rmse'])


--- KNNBasic Best Results ---
Найкращий RMSE: 0.9991593502885868
Найкращі параметри для RMSE: {'k': 20, 'min_k': 1, 'sim_options': {'name': 'msd', 'user_based': True}}


In [13]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
print("--- SVD Best Results ---")
print("Найкращий RMSE (SVD):", gs_svd.best_score["rmse"])
print("Найкращий MAE (SVD):", gs_svd.best_score["mae"])
print("Найкращі параметри (SVD):", gs_svd.best_params["rmse"])

print("\n--- KNNBasic Best Results ---")
print("Найкращий RMSE (KNN):", gs_knn.best_score["rmse"])
print("Найкращий MAE (KNN):", gs_knn.best_score["mae"])
print("Найкращі параметри (KNN):", gs_knn.best_params["rmse"])

--- SVD Best Results ---
Найкращий RMSE (SVD): 0.9303076413508009
Найкращий MAE (SVD): 0.736090855107022
Найкращі параметри (SVD): {'n_factors': 100, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}

--- KNNBasic Best Results ---
Найкращий RMSE (KNN): 0.9991593502885868
Найкращий MAE (KNN): 0.7893489041393392
Найкращі параметри (KNN): {'k': 20, 'min_k': 1, 'sim_options': {'name': 'msd', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
| :--| :--:| :--:| :--|
| SVD | 0.9303 | 0.7361 | n_factors: 100, n_epochs: 30, lr_all: 0.01, reg_all: 0.1 |
| KNNBasic | 0.9992 | 0.7893 | k: 20, min_k: 1, sim_options: {'name': 'msd', 'user_based': True} |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._



За результатами крос-валідації модель SVD показала кращу точність із меншим значенням RMSE (0.9303 проти 0.9992 у KNNBasic) та MAE (0.7361 проти 0.7893), при цьому найкращими параметрами для SVD стали `n_factors: 100`, `n_epochs: 30`, `lr_all: 0.01` та `reg_all: 0.1`, а для KNNBasic - `k: 20`, `min_k: 1` із мірою схожості `msd` та `user_based: True`.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [14]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:

# TODO 6.1
best_svd_params = gs_svd.best_params['rmse']
best_knn_params = gs_knn.best_params['rmse']

# TODO 6.2
trainset = train_data.build_full_trainset()

# TODO 6.3
# Для відтворюваності задайте random_state=42 для SVD.
best_svd = SVD(**best_svd_params, random_state=42)
best_svd.fit(trainset)

best_knn = KNNBasic(**best_knn_params)
best_knn.fit(trainset)

print("Моделі створено та навчено на повному trainset.")

Computing the msd similarity matrix...
Done computing similarity matrix.
Моделі створено та навчено на повному trainset.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [15]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
# TODO 7.1
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

# TODO 7.2
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

In [16]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame([
    {"Model": "SVD", "RMSE": svd_rmse, "MAE": svd_mae},
    {"Model": "KNNBasic", "RMSE": knn_rmse, "MAE": knn_mae}
])

results = results.sort_values(by="RMSE").reset_index(drop=True)
display(results)

,Model,RMSE,MAE
0,SVD,0.911756,0.72074
1,KNNBasic,0.971972,0.76518


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
| :--| :--:| :--:|
| SVD | 0.9118 | 0.7207 |
| KNNBasic | 0.9720 | 0.7652 |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

Оцінки на незалежній тестовій вибірці покращилися порівняно з результатами крос-валідації (напр., Test RMSE для SVD знизився з ~0.9303 до ~0.9118), що є нормальним явищем бо фінальні моделі були навчені на всій повноцінній навчальній вибірці (trainset), отримавши більше даних для узагальнення. При цьому модель SVD знову показує вищу точність ніж KNNBasic як на крос-валідації так і на тестовому наборі.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [20]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

In [22]:
# Ваш код:

# TODO 8.1
best_model = best_svd

# TODO 8.2 & 8.3
trainset = best_model.trainset

try:
    inner_uid = trainset.to_inner_uid(user_id)
except ValueError:
    print(f"Користувач з ID {user_id} відсутній у trainset!")
    inner_uid = None

if inner_uid is not None:
    # фільми які користувач вже оцінив
    rated_items = set([item_id for (item_id, rating) in trainset.ur[inner_uid]])

    # фільми у trainset (у внутрішніх ID)
    all_item_inner_ids = trainset.all_items()

    # неоцінені фільми, прогноз
    predictions = []
    for inner_iid in all_item_inner_ids:
        if inner_iid not in rated_items:
            raw_iid = trainset.to_raw_iid(inner_iid)
            # TODO 8.4 — прогноз за допомогою .predict()
            pred = best_model.predict(uid=user_id, iid=raw_iid)
            predictions.append((raw_iid, pred.est))

    # Створення DataFrame, сортуєвання за прогнозованим рейтингом, топ-10
    recommendations = pd.DataFrame(predictions, columns=["item_id", "predicted_rating"])
    recommendations = recommendations.sort_values(by="predicted_rating", ascending=False).head(10).reset_index(drop=True)

    print(f"--- Топ-10 рекомендацій для користувача {user_id} ---")
    display(recommendations)

--- Топ-10 рекомендацій для користувача 196 ---


,item_id,predicted_rating
0,1169,4.486375
1,1449,4.470392
2,318,4.467523
3,64,4.432005
4,320,4.381898
5,427,4.371729
6,114,4.339296
7,603,4.331796
8,867,4.276048
9,480,4.272109


## 9. Перевірка виконання

Позначте після реалізації:

- [x] Дані `ml-100k` завантажено та перевірено.
- [x] Є розбиття 80/20 з фіксованим `random_state`.
- [x] `Reader`, `train_data` і `testset` створені коректно.
- [x] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [x] Для SVD і KNNBasic показано параметри та результати CV.
- [x] Моделі навчено на повній навчальній частині.
- [x] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [x] Наведено порівняльну таблицю та інтерпретацію.
- [x] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [x] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?

   **Відповідь:** обрано SVD оскільки вона показала меншу похибку


2. Які найкращі гіперпараметри отримано для обох моделей?

   **Відповідь:** SVD: n_factors: 100, n_epochs: 30, lr_all: 0.01, reg_all: 0.1.  
    KNNBasic: k: 20, min_k: 1, sim_options: {'name': 'msd', 'user_based': True}


3. Які підсумкові значення RMSE та MAE на тестовій вибірці?

    **Відповідь:** SVD: RMSE ≈ 0.9118, MAE ≈ 0.7207;

    KNNBasic: RMSE ≈ 0.9720, MAE ≈ 0.7652.

4. Чи є різниця між результатами CV та тестування?

    **Відповідь:** Так, на тесті показники трохи кращі через те що фінальні моделі навчалися на всій повноцінній вибірці (trainset)

5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

    **Відповідь:** Основне обмеження це (cold start): модель не може ефективно прогнозувати рейтинги або формувати рекомендації для нових користувачів чи фільмів, які не мають історії оцінок у наборі даних.

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

Було побудовано, налаштовано та порівняно дві моделі колаборативної фільтрації (SVD та KNNBasic) на базі набору даних ml-100k. За результатами крос-валідації та пошуку гіперпараметрів перевагу отримала модель SVD з показником CV RMSE 0.9303 проти 0.9992 у KNNBasic. Після навчання на всій навчальній вибірці моделі перевірено на незалежному тестовому наборі, де SVD підтвердила вищу точність із результатами Test RMSE 0.9118 та Test MAE 0.7207. Для порівняння, алгоритм KNNBasic на тестових даних показав вищу похибку: Test RMSE 0.9720 та Test MAE 0.7652. На основі найкращої моделі SVD було сформовано топ-10 персоналізованих рекомендацій для користувача 196 (з найвищим прогнозованим рейтингом 4.4864 для фільму 1169), виключивши вже оцінені ним стрічки. Таким чином, матрична факторизація виявилася ефективнішою для даної задачі, хоча підхід зберігає типове обмеження у вигляді проблеми (cold start) для нових користувачів та фільмів.

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

In [23]:
class SimpleMeanRecommender:
    def __init__(self):
        self.global_mean = 0
        self.user_means = {}
        self.item_means = {}

    def fit(self, trainset):
        ratings = [rating for (_, _, rating) in trainset.all_ratings()]
        self.global_mean = np.mean(ratings)

        for u in trainset.all_users():
            user_ratings = [rating for (_, rating) in trainset.ur[u]]
            if user_ratings:
                self.user_means[u] = np.mean(user_ratings)
            else:
                self.user_means[u] = self.global_mean

        for i in trainset.all_items():
            item_ratings = [rating for (_, rating) in trainset.ir[i]]
            if item_ratings:
                self.item_means[i] = np.mean(item_ratings)
            else:
                self.item_means[i] = self.global_mean

        return self

    def predict(self, uid, iid):
        # Прогноз на основі глобального середнього (базовий алгоритм)
        pred_rating = self.global_mean
        return type('Prediction', (object,), {'est': pred_rating})()

# Створення, навчання та перевірка власної моделі
my_recommender = SimpleMeanRecommender()
my_recommender.fit(trainset)

# Приклад прогнозу для користувача '196' та фільму '318'
pred = my_recommender.predict(uid='196', iid='318')
print(f"Прогнозований рейтинг власним алгоритмом: {pred.est:.4f}")

Прогнозований рейтинг власним алгоритмом: 3.5313
